# EDA — Trayectorias de Tetris

Los datos no son un dataset externo: son **trayectorias generadas por interacción** del agente con el entorno (una fila por colocación, vía el pipeline de generación).

Dos partes:
1. **EDA base** con `RandomAgent`: duración de episodios, recompensas, líneas y features.
2. **Comparación de agentes**: random, heurístico, TD y CEM (TD/CEM se incluyen solo si existe su `.npy` de pesos).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tetris_rl.agents import RandomAgent, HeuristicAgent, LinearAgent
from tetris_rl.data.generate import generate
from tetris_rl.features import FEATURE_NAMES

SEED = 0
EPISODES = 30
MAX_STEPS = 200          # tope de colocaciones por partida (agentes fuertes no terminan solos)
RAW = Path('../data/raw')
MODELS = Path('../models')

In [ ]:
def make_agent(name):
    """Agente por nombre. TD/CEM cargan sus pesos; None si el .npy no existe."""
    if name == 'random':
        return RandomAgent(seed=SEED)
    if name == 'heuristic':
        return HeuristicAgent()
    wpath = MODELS / f'{name}_weights.npy'
    if wpath.exists():
        return LinearAgent(np.load(wpath))
    print(f'(sin pesos para {name}: {wpath} no existe, se omite)')
    return None


def by_episode(df):
    """Agrega a nivel episodio: colocaciones, líneas y recompensa totales."""
    return df.groupby('episode').agg(
        steps=('step', 'size'),
        lines=('lines_cleared', 'sum'),
        reward=('reward', 'sum'),
    )

## 1. EDA base — `RandomAgent`

In [ ]:
rnd_csv = generate(make_agent('random'), episodes=EPISODES, seed=SEED,
                   out_dir=str(RAW), run_name='random', max_steps=MAX_STEPS)
df = pd.read_csv(rnd_csv)
print(f'{len(df)} transiciones en {df.episode.nunique()} episodios')
df.head()

In [ ]:
per_ep = by_episode(df)
per_ep.describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].hist(per_ep['steps'], bins=20); axes[0].set_title('Duración de episodios (colocaciones)')
axes[1].hist(per_ep['lines'], bins=20); axes[1].set_title('Líneas por episodio')
axes[2].hist(per_ep['reward'], bins=20); axes[2].set_title('Recompensa por episodio')
plt.tight_layout()

**Hallazgo esperado:** con `RandomAgent` casi no se completan líneas y los episodios son cortos. No es un bug: es la recompensa dispersa del problema (motiva usar agentes que aprenden).

### Estadísticas de las 6 features

In [ ]:
df[list(FEATURE_NAMES)].describe()

In [ ]:
df[list(FEATURE_NAMES)].hist(figsize=(12, 7), bins=20)
plt.tight_layout()

## 2. Comparación de agentes

Genera partidas de evaluación de cada agente y compara líneas y duración. TD y CEM usan `LinearAgent` con sus pesos entrenados (juego greedy).

In [ ]:
rows = []
for name in ['random', 'heuristic', 'td', 'cem']:
    agent = make_agent(name)
    if agent is None:
        continue
    csv = generate(agent, episodes=EPISODES, seed=SEED, out_dir=str(RAW),
                   run_name=f'eval_{name}', max_steps=MAX_STEPS)
    ep = by_episode(pd.read_csv(csv))
    rows.append({'agente': name,
                 'lineas_medias': ep['lines'].mean(),
                 'duracion_media': ep['steps'].mean(),
                 'lineas_totales': ep['lines'].sum()})
comp = pd.DataFrame(rows).set_index('agente')
comp

In [ ]:
comp[['lineas_medias', 'duracion_media']].plot.bar(
    subplots=True, figsize=(8, 6), legend=False)
plt.tight_layout()

**Notas para el informe**

- Las cifras de TD/CEM dependen de pesos entrenados; re-ejecutar este notebook tras regenerar los `.npy` con la config final.
- La comparación es justa en duración/líneas (no en la recompensa moldeada, que mezcla líneas² con la penalización de game-over).